<a href="https://colab.research.google.com/github/jumafernandez/modelado-gestion-datos-untref/blob/main/codigo/python_postgresql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conectar Python con PostgreSQL

**Modelado y Gestión de Datos - UNTREF - 2026**

En este notebook nos conectamos desde Python a un servidor PostgreSQL del curso, le enviamos consultas SQL y recibimos los resultados como *data frames*. El servidor tiene dos bases de solo lectura:

- `w3schools`, la base de práctica del TP0302.
- `eph`, con los microdatos de la EPH que armamos en el cuaderno [De la EPH a PostgreSQL](https://jumafernandez.github.io/modelado-gestion-datos-untref/guias/cuaderno-eph/).

Cada celda se ejecuta con **Shift + Enter** o con el botón ▶ de su izquierda.

## 1. Los paquetes

pandas trae los resultados como data frames, SQLAlchemy maneja la conexión y psycopg2 es el driver de PostgreSQL. Colab ya los trae instalados.

In [ ]:
import pandas as pd
from sqlalchemy import URL, create_engine

## 2. Conectarse a la base

`create_engine()` abre la conexión con los mismos datos que usamos en pgAdmin: el servidor, el puerto, la base, el usuario y la clave. Antes de ejecutar la celda hay que reemplazar `CLAVE` por la clave del usuario `alumno`, que está en el campus.

In [ ]:
url = URL.create("postgresql+psycopg2",
                 host="170.210.103.79",
                 port=80,
                 database="w3schools",
                 username="alumno",
                 password="CLAVE")
engine = create_engine(url)

# las tablas de la base
pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'", engine)

## 3. Una consulta, un data frame

`pd.read_sql()` envía una consulta SQL y devuelve el resultado como data frame. Es la misma consulta que escribiríamos en pgAdmin; esta responde la primera pregunta del Anexo I del TP0302. PostgreSQL guarda los nombres de tablas y columnas en minúscula, y así aparecen en el data frame.

In [ ]:
productos = pd.read_sql("""
SELECT productname, price
FROM products
WHERE price > 50
ORDER BY price DESC
""", engine)

productos

## 4. Una consulta agrupada y un gráfico

Conviene que PostgreSQL resuelva las juntas y los agrupamientos, y que a Python llegue solo el resultado: acá, una fila por categoría con el total vendido.

In [ ]:
ventas = pd.read_sql("""
SELECT c.categoryname AS categoria,
       SUM(od.quantity * p.price) AS ventas
FROM orderdetails od
INNER JOIN products p   ON od.productid = p.productid
INNER JOIN categories c ON p.categoryid = c.categoryid
GROUP BY c.categoryname
ORDER BY ventas DESC
""", engine)

ventas

In [ ]:
import matplotlib.pyplot as plt

ax = ventas.sort_values("ventas").plot.barh(x="categoria", y="ventas", color="#E08A1E", legend=False)
ax.set_title("Ventas por categoría")
ax.set_xlabel("Total vendido")
ax.set_ylabel("")
plt.show()

## 5. Cerrar la conexión

Al terminar, se cierran las conexiones con el servidor.

In [ ]:
engine.dispose()

---

# La EPH desde Python

Ahora, el mismo camino con datos reales: la base `eph`, con los microdatos de la Encuesta Permanente de Hogares de INDEC desde 2018. El análisis usa pandas, NumPy, Matplotlib y seaborn, que Colab ya trae instalados.

In [ ]:
import numpy as np
import seaborn as sns

url = URL.create("postgresql+psycopg2",
                 host="170.210.103.79",
                 port=80,
                 database="eph",
                 username="alumno",
                 password="CLAVE")
engine = create_engine(url)

pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'", engine)

## Las personas de un trimestre

De una tabla grande conviene pedir solo lo que se va a usar: `personas` tiene más de un millón y medio de filas. La consulta une cada persona con su hogar (para el ponderador) y llega hasta la región. Trae el primer trimestre de 2026; para analizar otro, se cambian los valores del `WHERE`.

In [ ]:
consulta_personas = """
SELECT p.sexo,
       p.edad,
       p.id_condicion_actividad,
       r.nombre AS region,
       h.pondera
FROM personas p
INNER JOIN hogares h     ON p.codusu = h.codusu AND p.anio = h.anio
                        AND p.trimestre = h.trimestre AND p.nro_hogar = h.nro_hogar
INNER JOIN viviendas v   ON h.codusu = v.codusu
INNER JOIN aglomerados a ON v.id_aglomerado = a.id
INNER JOIN regiones r    ON a.id_region = r.id
WHERE p.anio = 2026 AND p.trimestre = 1
"""

personas = pd.read_sql(consulta_personas, engine)

personas.info()
print("personas encuestadas:", len(personas))
print("población que representan:", personas["pondera"].sum())

## Tasas del mercado de trabajo por región

Condición de actividad: 1 ocupado, 2 desocupado, 3 inactivo, 4 menor de 10 años. Las tasas suman el ponderador, como las de INDEC:

- actividad = (ocupados + desocupados) / población
- empleo = ocupados / población
- desocupación = desocupados / (ocupados + desocupados)

In [ ]:
personas["ocupados"] = personas["pondera"].where(personas["id_condicion_actividad"] == 1, 0)
personas["desocupados"] = personas["pondera"].where(personas["id_condicion_actividad"] == 2, 0)

tasas_region = personas.groupby("region")[["pondera", "ocupados", "desocupados"]].sum()
activos = tasas_region["ocupados"] + tasas_region["desocupados"]
tasas_region["tasa_actividad"] = (100 * activos / tasas_region["pondera"]).round(1)
tasas_region["tasa_empleo"] = (100 * tasas_region["ocupados"] / tasas_region["pondera"]).round(1)
tasas_region["tasa_desocupacion"] = (100 * tasas_region["desocupados"] / activos).round(1)

tasas_region[["tasa_actividad", "tasa_empleo", "tasa_desocupacion"]].sort_values("tasa_desocupacion", ascending=False)

## La serie de la desocupación

Para recorrer todos los trimestres no hace falta traer el millón y medio de personas: PostgreSQL agrupa y calcula la tasa, y llega una fila por trimestre.

In [ ]:
consulta_serie = """
SELECT p.anio,
       p.trimestre,
       ROUND(100.0 * SUM(CASE WHEN p.id_condicion_actividad = 2 THEN h.pondera ELSE 0 END)
                   / SUM(CASE WHEN p.id_condicion_actividad IN (1, 2) THEN h.pondera ELSE 0 END),
             1) AS tasa_desocupacion
FROM personas p
INNER JOIN hogares h ON p.codusu = h.codusu AND p.anio = h.anio
                    AND p.trimestre = h.trimestre AND p.nro_hogar = h.nro_hogar
GROUP BY p.anio, p.trimestre
ORDER BY p.anio, p.trimestre
"""

serie = pd.read_sql(consulta_serie, engine)
serie

In [ ]:
# Para el eje del gráfico: 2026.00 es el primer trimestre, 2026.25 el segundo...
serie["periodo"] = serie["anio"] + (serie["trimestre"] - 1) / 4

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(serie["periodo"], serie["tasa_desocupacion"], color="#E08A1E", linewidth=2)
ax.scatter(serie["periodo"], serie["tasa_desocupacion"], color="#1C1A3A", s=12, zorder=3)
ax.set_xticks(range(2018, 2027))
ax.set_title("Tasa de desocupación por trimestre")
ax.set_ylabel("% de la población económicamente activa")
plt.show()

## Estructura por edad y sexo

Con el data frame de las personas del trimestre, agrupado en tramos de cinco años.

In [ ]:
etiquetas_edad = [f"{edad}-{edad + 4}" for edad in range(0, 80, 5)] + ["80 y más"]
personas["grupo_edad"] = pd.cut(personas["edad"], bins=list(range(0, 85, 5)) + [np.inf],
                                right=False, labels=etiquetas_edad)
personas["sexo_texto"] = personas["sexo"].map({1: "Varones", 2: "Mujeres"})

piramide = personas.pivot_table(index="grupo_edad", columns="sexo_texto", values="pondera",
                                aggfunc="sum", observed=False)
piramide = 100 * piramide / piramide.to_numpy().sum()


def sin_signo(valor, posicion):
    return f"{abs(valor):g}"


fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(etiquetas_edad, -piramide["Varones"], color="#1C1A3A", label="Varones")
ax.barh(etiquetas_edad, piramide["Mujeres"], color="#E08A1E", label="Mujeres")
ax.xaxis.set_major_formatter(sin_signo)
ax.set_title("Estructura por edad y sexo, primer trimestre de 2026")
ax.set_xlabel("% de la población")
ax.legend()
plt.show()

## Ingreso per cápita familiar por región

La vista `ingresos_hogares` del cuaderno ya lo calcula. Para los ingresos, INDEC usa el ponderador PONDIH; los hogares que no respondieron tienen PONDIH = 0 y quedan afuera. La mediana ponderada es el primer valor en el que el peso acumulado llega a la mitad.

In [ ]:
consulta_ingresos = """
SELECT r.nombre AS region,
       i.ingreso_per_capita,
       i.pondih
FROM ingresos_hogares i
INNER JOIN viviendas v   ON i.codusu = v.codusu
INNER JOIN aglomerados a ON v.id_aglomerado = a.id
INNER JOIN regiones r    ON a.id_region = r.id
WHERE i.anio = 2026 AND i.trimestre = 1 AND i.pondih > 0
"""

ingresos = pd.read_sql(consulta_ingresos, engine)


def mediana_ponderada(x, w):
    orden = np.argsort(x.to_numpy())
    valores = x.to_numpy()[orden]
    peso_acumulado = np.cumsum(w.to_numpy()[orden])
    return valores[np.searchsorted(peso_acumulado, peso_acumulado[-1] / 2)]


filas = []
for region, grupo in ingresos.groupby("region"):
    filas.append({
        "region": region,
        "hogares_encuestados": len(grupo),
        "media": round(np.average(grupo["ingreso_per_capita"], weights=grupo["pondih"])),
        "mediana": round(mediana_ponderada(grupo["ingreso_per_capita"], grupo["pondih"])),
    })

pd.DataFrame(filas).sort_values("mediana", ascending=False)

In [ ]:
# La distribución es muy asimétrica: el eje va en escala logarítmica.
# Los hogares con ingreso cero no tienen logaritmo y no se grafican.
positivos = ingresos[ingresos["ingreso_per_capita"] > 0]

g = sns.displot(data=positivos, x="ingreso_per_capita", weights="pondih", col="region", col_wrap=2,
                kind="kde", log_scale=True, common_norm=False, fill=True, color="#E08A1E",
                height=2.4, aspect=1.7)
g.set_titles("{col_name}")
g.set_axis_labels("Pesos por mes (escala logarítmica)", "")
g.set(yticks=[])
g.figure.suptitle("Ingreso per cápita familiar, primer trimestre de 2026", y=1.03)
plt.show()

## Cerrar la conexión

In [ ]:
engine.dispose()

---

## La contraseña, fuera del código

En este notebook la clave se escribe en el código porque el usuario `alumno` es de solo lectura y su clave es pública para el curso. Con credenciales propias no conviene: un notebook con la contraseña escrita termina compartido por mail o subido a un repositorio. En Colab, la práctica es guardarla en los **Secretos** del notebook:

1. En el panel de la izquierda, abrir el ícono de la llave (*Secretos*).
2. Agregar un secreto con el nombre `PGPASSWORD` y la clave como valor, y activar el acceso del notebook.
3. Leerlo con `userdata.get()` al conectarse, como en la celda de abajo.

El secreto queda asociado a tu cuenta: no viaja con el notebook cuando lo compartís.

In [ ]:
from google.colab import userdata

url = URL.create("postgresql+psycopg2",
                 host="170.210.103.79",
                 port=80,
                 database="w3schools",
                 username="alumno",
                 password=userdata.get("PGPASSWORD"))
engine = create_engine(url)
pd.read_sql("SELECT count(*) AS productos FROM products", engine)